# 策略概述

**SSD Rolling** 是配對交易最經典的**距離法**基準：在同一產業內尋找歷史走勢**最接近**的兩檔股票組成配對，當兩者價差偏離常態時進場、回歸常態時獲利了結。它是後續所有機器學習分組法要超越的**傳統對照組**。

核心直覺：兩檔長期同漲同跌的股票，其價差應在穩定區間內波動（均值回歸）；距離（SSD）越小代表兩者走勢越同步、越可能存在這種可交易的穩定關係。


# 策略架構

形成期管線由左到右分四層處理。**本策略定位為傳統基準，四層皆採最經典方法**；後續機器學習策略只替換「分組」一層，其餘完全相同——這是命題 1 的乾淨單變因對照設計。

```{mermaid}
flowchart LR
  P["形成期日價格<br/>252 日"] --> G["分組<br/>GICS 產業"]
  G --> R["排序<br/>最小 SSD 距離"]
  R --> F["篩選<br/>共整合 + 半衰期 + Hurst"]
  F --> T["Top N 配對<br/>→ 交易期"]
```

| 層 | 本策略採用 | 用途 |
| :--- | :--- | :--- |
| 分組 | GICS 產業分類 | 限縮候選在同產業內（基本面相近，較可能有長期均衡） |
| 排序 | 最小 SSD 距離 | 挑出歷史走勢最接近的股票對 |
| 篩選 | 共整合檢定 + 半衰期 + Hurst | 確認價差確實會均值回歸、非隨機漂移 |
| 交易 | Z-Score 回歸 | 價差偏離進場、回歸均值出場 |


# 參考文獻與引用對應

本策略的每個設計環節皆有明確文獻依據，逐一標示「參考了文獻的哪個部分」與「為何參考」。


## 文獻 1：Gatev, Goetzmann & Rouwenhorst (2006)

> Gatev, E., Goetzmann, W. N., & Rouwenhorst, K. G. (2006). Pairs trading: Performance of a relative value arbitrage rule. *Review of Financial Studies, 19*(3), 797–827.

**參考部分**：

- 第 2 節配對形成方法——以**標準化價格序列間的距離平方和（SSD）最小**作為配對排序準則
- 形成期／交易期分離的雙階段回測架構（12 個月形成、6 個月交易）

**為何參考**：

- 這是配對交易距離法的**開山文獻**，SSD 排序是本策略的核心排序準則，直接沿用其「距離越小 → 歷史走勢越相近 → 越可能存在均衡關係」的邏輯
- 本策略的形成期 252 日／交易期 126 日視窗設計即對應其 12 個月／6 個月架構



## 文獻 2：Engle & Granger (1987)

> Engle, R. F., & Granger, C. W. J. (1987). Co-integration and error correction: Representation, estimation, and testing. *Econometrica, 55*(2), 251–276.

**參考部分**：

- 共整合的正式定義：兩個 I(1) 序列若存在線性組合為 I(0)，則兩序列共整合
- 兩步驟檢定程序：先以 OLS 估計均衡關係，再對殘差做 **ADF 單根檢定**

**為何參考**：

- SSD 距離只保證「歷史走勢相近」，不保證價差會回歸——必須以共整合檢定確認兩股間存在**統計上的長期均衡關係**
- 本策略統計篩選第一道的 ADF 檢定（對 OLS 殘差 spread 檢定單根）即為其兩步驟程序的直接實作



## 文獻 3：Krauss, Do & Huck (2016)

> Krauss, C., Do, X. A., & Huck, N. (2016). The profitability of pairs trading strategies: Distance, cointegration and copula methods. *European Journal of Operational Research*.

**參考部分**：

- 對距離法的系統性檢視：純距離排序選出的配對，其價差**均值回歸強度參差**，需輔以額外統計篩選
- **OU（Ornstein-Uhlenbeck）半衰期**與 **Hurst 指數**作為均值回歸品質的量化指標

**為何參考**：

- 本策略統計篩選第二道（半衰期範圍限制）與第三道（Hurst < 0.5）的設計依據——確保選出的配對不僅共整合，且**回歸速度落在交易期可實現的時間尺度內**
- 半衰期上限設為交易期長度的 1/3（126/3 = 42 日），即出自「交易期內須有足夠回歸次數」的原則



## 文獻 4：Zhu (2024)

> Zhu, X. (2024). Examining Pairs Trading Profitability. Senior Essay, Department of Economics, Yale University.

**參考部分**：

- 以 2003–2023 年資料**複製 Gatev et al. (2006) 距離法**：標準化價格 + 歐氏距離（SSD）配對選取，報告年化超額報酬 6.2%、Sharpe 1.35
- 對「先距離初篩、再做共整合檢定」兩步驟選取流程的計算成本分析：全市場 $N(N-1)/2$ 對逐一共整合檢定不可行，需先以距離縮小候選集

**為何參考**：

- 提供 SSD 距離法在**近二十年資料上仍然有效**的實證佐證，支持本策略以 SSD 作為核心排序準則的現代適用性
- 本策略階段 4 的「SSD 初篩 → 統計檢定只做前 $\max(200,\ top\_n \times 15)$ 組」設計，與其成本分析的結論一致



# 各階段行為

策略在每個滾動形成窗（252 交易日，每 21 日滾動一次）內依序執行以下六個階段。


## 階段 1：資料範圍界定與產業分組

**輸入**：形成窗內全部 S&P 500 歷史成分股的日收盤價矩陣。

**行為**：

1. 依 GICS 產業分類（`sector_mapping`）將股票分組，配對搜尋**只在同產業內部進行**
2. 產業標記為 `Unknown`（無法對應 GICS 分類）的股票整組跳過，不參與配對
3. 股票數不足 `min_tickers_for_pairing`（= 2）的產業跳過

**設計理由**：同產業股票暴露於相同的產業景氣、法規與供需因子，價格間的均衡關係具備經濟基礎，而非純統計巧合。


## 階段 2：對數價格 Z-Score 標準化

對形成窗內每支股票 $i$ 的價格序列 $P_{i,t}$：

$$P'_{i,t} = \frac{\ln P_{i,t} - \mu_{\ln P_i}}{\sigma_{\ln P_i} + \varepsilon}$$

其中 $\mu_{\ln P_i}$、$\sigma_{\ln P_i}$ 為該股**形成窗內**對數價格的均值與標準差，$\varepsilon = 10^{-12}$ 防除零；取對數前價格先以 $\max(P, 10^{-8})$ 下限保護。

**目的**：

- 取對數：把價格變動轉為報酬尺度，消除高低價股的絕對價差
- Z-Score：使每支股票的標準化序列均值為 0、標準差為 1——SSD 比較的是**走勢形狀**而非價格水準或波動幅度

標準化用的 $\mu_{\ln P_i}$、$\sigma_{\ln P_i}$ 會隨配對一併輸出（`Log_Mean_A/B`、`Log_Std_A/B`），交易期沿用**同一組**形成期統計量重建 spread，保證兩階段座標一致。


## 階段 3：SSD 計算與 OLS 對沖比例（批次矩陣運算）

對同產業內每一對股票 $(A, B)$：

**SSD 距離**（`scipy.spatial.distance.pdist`，`sqeuclidean` 度量）：

$$\text{SSD}_{A,B} = \sum_{t=1}^{F} \left(P'_{A,t} - P'_{B,t}\right)^2$$

**OLS 對沖比例**（由整個產業的協方差矩陣一次求得，免逐對回歸）：

$$\beta_{A,B} = \frac{\text{Cov}(P'_A,\ P'_B)}{\text{Var}(P'_B)}$$

**價差序列（spread）**：

$$\epsilon_t = P'_{A,t} - \beta_{A,B} \cdot P'_{B,t}$$

註：$\text{Var}(P'_B) \le 10^{-8}$ 時 $\beta$ 設為 0（該對不會通過後續檢定）；標準化空間中 OLS 不含截距項。


## 階段 4：候選初篩

全部配對依 SSD **升序**排列後，只取前

$$N_{cand} = \max(200,\ top\_n \times 15)$$

組進入統計檢定。

**設計理由**：ADF／半衰期／Hurst 屬於逐對的慢速計算；SSD 最小的候選對才有機會入選最終名單，對距離過遠的配對做完整檢定是無效計算。此初篩把統計檢定次數從 $O(N^2)$ 壓到常數規模。


## 階段 5：三道統計篩選

候選對依 SSD 由小到大逐一檢定，任一道未通過即淘汰。

### 第一道：ADF 共整合檢定（依據：Engle & Granger 1987）

對 spread $\epsilon_t$ 做 ADF 單根檢定（`_utils._adf_stat`，`max_lags=1`，`regression="n"` 無截距無趨勢——因 spread 由標準化序列構成，理論均值即為 0）：

$$H_0: \epsilon_t \sim I(1)\ (\text{隨機漫步}) \qquad \text{要求 } p < 0.05 \text{ 拒絕 } H_0$$

### 第二道：OU 半衰期（依據：Krauss et al. 2016）

以離散 OU 過程估計回歸速度，對 $\Delta \epsilon_t = c + \lambda\, \epsilon_{t-1} + u_t$ 做最小平方估計：

$$\lambda < 0 \ \text{（必要條件，否則無回歸傾向）}, \qquad HL = \frac{-\ln 2}{\lambda}$$

$$\text{要求 } 1 \le HL \le \frac{T_{trading}}{3} = 42 \text{ 日}$$

- 下限 1 日：排除日內噪音級別的假回歸
- 上限 42 日：交易期 126 日內至少可容納約 3 個完整回歸半衰期

### 第三道：Hurst 指數（依據：Krauss et al. 2016）

對 spread 直接做 R/S 分析（`already_stationary=True`，不再差分）：

$$H < 0.50 \quad (H < 0.5 \text{ 為均值回歸；} H = 0.5 \text{ 隨機漫步；} H > 0.5 \text{ 趨勢延續})$$

通過三道檢定的配對達 $top\_n \times 5$ 組時提前停止檢定（已足夠產生最終名單）。


## 階段 6：配對選取與參數輸出

通過全部檢定的配對依 SSD 升序取前 `top_n` 組，每組輸出下列欄位供交易期使用：

| 欄位 | 內容 | 交易期用途 |
| :--- | :--- | :--- |
| `Ticker_A` / `Ticker_B` | 配對股票代碼 | 建倉標的 |
| `Sector` | GICS 產業 | 產業分散控管 |
| `SSD` | 距離值 | 排序依據（記錄用） |
| `Rank` | 名次（1 起算） | 記錄用 |
| `Hedge_Ratio` | OLS 對沖比例 $\beta$ | spread 重建與部位配重 |
| `Spread_Mean` / `Spread_Std` | 形成期 spread 均值 $\mu_\epsilon$／標準差 $\sigma_\epsilon$ | Z-Score 分母與中心 |
| `Log_Mean_A/B`、`Log_Std_A/B` | 形成期對數價格均值／標準差 | 交易期標準化座標 |


## 階段 7：交易期的參數使用方式

交易期（126 日）每日以**形成期凍結的統計量**重建 spread 與 Z-Score：

$$P'_{i,t} = \frac{\ln P_{i,t} - \texttt{Log\_Mean}_i}{\texttt{Log\_Std}_i}, \qquad
\text{Spread}_t = P'_{A,t} - \texttt{Hedge\_Ratio} \cdot P'_{B,t}$$

$$Z_t = \frac{\text{Spread}_t - \texttt{Spread\_Mean}}{\texttt{Spread\_Std}}$$

形成期統計量在整個交易期**保持不變**——交易訊號完全建立在「形成期建立的均衡關係是否延續」之上，不摻入交易期內的資訊（無前視）。交易決策細節見交易期筆記本 `trading/zscore_trading.ipynb`。


# 參數總表

| 參數 | 值 | 對應階段 | 說明 |
| :--- | :---: | :--- | :--- |
| 形成窗長度 $F$ | 252 交易日 | 全流程輸入 | 約一年的觀察窗 |
| 滾動步長 | 21 交易日 | 全流程輸入 | 約一個月推進一次 |
| 每期配對數 | 網格 [1, 3, 5, 10, 20] | 排序（選取） | 依 SSD 由小到大取前幾組 |
| 共整合顯著水準 | 0.05 | 篩選 | 價差通過共整合檢定的門檻 |
| 半衰期範圍 | $[1,\ 42]$ 日 | 篩選 | 回歸速度須落在合理區間（交易期 126÷3） |
| Hurst 上限 | 0.50 | 篩選 | 均值回歸判準（<0.5 才視為回歸型） |
| 候選初篩上限 | $\max(200,\ 每期配對數 \times 15)$ | 排序（計算預算） | 慢速統計檢定的候選上限 |
| 提前停止門檻 | 每期配對數 $\times 5$ | 篩選 | 通過檢定配對數達此值即停止 |
